# Vigil — Notebook 01: Ingestión de Datos y Normalización a Silver

Este cuaderno implementa la Fase 1 del pipeline: Ingestión de los archivos de Facebook crudos de Apify y su mapeo al esquema de la capa Silver.

In [ ]:
import polars as pl
import yaml
from pathlib import Path
import sys

# Agregar src/ al path por seguridad si es necesario
sys.path.append('..')
from src.processing.quality_gate import mapear_esquema_crudo

print("Polars version:", pl.__version__)

## 1. Cargar Configuración Maestra y Catálogo de Seguidores

In [ ]:
with open("../config/config.yaml") as f:
    config = yaml.safe_load(f)

print("Candidatos configurados:", [c['nombre'] for c in config['candidatos']])
print("Catálogo de seguidores:", config['catalogo_seguidores'])

## 2. Cargar Datasets Crudos (Fase Raw)

In [ ]:
posts_raw = pl.read_ndjson("../data/raw/facebook/fb_posts_tepic_2026-02-20.jsonl")
comments_raw = pl.read_ndjson("../data/raw/facebook/fb_comments_tepic_2026-02-20.jsonl")

print("Posts crudos:", posts_raw.height)
print("Comentarios crudos:", comments_raw.height)

## 3. Mapear y Normalizar a Capa Silver

In [ ]:
posts_silver = mapear_esquema_crudo(posts_raw, config["catalogo_seguidores"], "DIFNay")
comments_silver = mapear_esquema_crudo(comments_raw, config["catalogo_seguidores"], "GeraldinePonceMexico")

print("Posts mapeados a Silver:")
print(posts_silver.head(3))
print("\nComentarios mapeados a Silver:")
print(comments_silver.head(3))

## 4. Persistir en la Capa Silver (Formato Parquet)

In [ ]:
Path("../data/silver/redes_sociales").mkdir(parents=True, exist_ok=True)

posts_silver.write_parquet("../data/silver/redes_sociales/fb_posts_silver.parquet")
comments_silver.write_parquet("../data/silver/redes_sociales/fb_comments_silver.parquet")

print("Fase 1 Completada: Datasets de Facebook guardados en silver/redes_sociales/")